# Step 7 – Attention U-Net (3 seeds) vs plain U-Net
Trains a U-Net with **scSE attention** in the decoder (spatial + channel attention, `decoder_attention_type='scse'`), in two versions:
- **C: attention, SAR only** (VV, VH)
- **D: attention, SAR + slope + HAND**

Seeds 0, 1, 2 with exactly the same settings as notebook 05, then compared with plain U-Nets A and B from notebook 05.

Note for the paper: scSE is a light attention block, not the attention-gate design of Oktay et al. (2018). Describe it as "U-Net with scSE decoder attention".

- Runtime: **GPU (T4)**, about 1–1.5 hours (6 training runs).
- **Resumable:** each finished run is saved to Drive right away. If Colab disconnects, just **Run all** again: finished runs are skipped.
- No Earth Engine needed: terrain chips are read from the cache made in notebook 02.
- Keep the browser tab open; free Colab may disconnect if the tab is idle for a long time.

In [ ]:
!pip -q install segmentation-models-pytorch rasterio
import os, glob, random, subprocess, numpy as np, pandas as pd, rasterio, torch, torch.nn as nn
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEV)

from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Paper2_flood'
CACHE = f'{BASE}/02_terrain/terrain_chips'
OUT = f'{BASE}/07_attention'
PREV = f'{BASE}/05_multiseed/runs'; os.makedirs(f'{OUT}/runs', exist_ok=True)
assert os.path.isdir(CACHE), 'terrain chips from notebook 02 not found - check the Drive account'
SEEDS = [0, 1, 2]
assert os.path.isdir(PREV), 'notebook 05 runs not found'

## 1. Data (same as notebook 02)

In [ ]:
ROOT = '/content/sen1floods11'; os.makedirs(ROOT, exist_ok=True)
B = 'gs://sen1floods11/v1.1'
if not os.path.exists(f'{ROOT}/LabelHand'):
    for p in ['data/flood_events/HandLabeled/S1Hand', 'data/flood_events/HandLabeled/LabelHand',
              'splits/flood_handlabeled']:
        subprocess.run(['gsutil', '-m', '-q', 'cp', '-r', f'{B}/{p}', f'{ROOT}/'], check=True)

def chip_ids(csv):
    return [f.replace('_S1Hand.tif', '') for f in pd.read_csv(csv, header=None)[0]]
S = f'{ROOT}/flood_handlabeled'
split = {k: chip_ids(f'{S}/flood_{k}_data.csv') for k in ['train', 'valid', 'test', 'bolivia']}
ALL = [c for v in split.values() for c in v]

def load(cid):
    with rasterio.open(f'{ROOT}/S1Hand/{cid}_S1Hand.tif') as s: x = s.read().astype('float32')
    with rasterio.open(f'{ROOT}/LabelHand/{cid}_LabelHand.tif') as l: y = l.read(1).astype('int64')
    x = np.clip(np.nan_to_num(x, nan=-50), -50, 1)
    t = np.nan_to_num(np.load(f'{CACHE}/{cid}.npy'))
    slope = t[0].copy()
    t = np.stack([np.clip(t[0], 0, 60), np.log1p(np.clip(t[1], 0, 200))])
    y[y < 0] = 255
    return np.concatenate([x, t]), y, slope

data = {c: load(c) for c in ALL}
tr = np.stack([data[c][0] for c in split['train']])
MEAN, STD = tr.mean((0, 2, 3)), tr.std((0, 2, 3)) + 1e-6; del tr

class Chips(Dataset):
    def __init__(self, ids, ch, train=False, crop=256):
        self.ids, self.ch, self.train, self.crop = ids, ch, train, crop
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        x, y, s = data[self.ids[i]]
        x = ((x - MEAN[:, None, None]) / STD[:, None, None])[self.ch]
        if self.train:
            r, c = np.random.randint(0, 512 - self.crop, 2)
            x, y, s = x[:, r:r+self.crop, c:c+self.crop], y[r:r+self.crop, c:c+self.crop], s[r:r+self.crop, c:c+self.crop]
            if random.random() < .5: x, y, s = x[:, :, ::-1], y[:, ::-1], s[:, ::-1]
            if random.random() < .5: x, y, s = x[:, ::-1], y[::-1], s[::-1]
        return torch.tensor(x.copy()), torch.tensor(y.copy()), torch.tensor(s.copy())
print({k: len(v) for k, v in split.items()})

## 2. Training and evaluation (identical to notebook 05, plus attention option)

In [ ]:
def scores(tp, fp, fn, tn):
    iou = tp / (tp + fp + fn + 1e-9); p = tp / (tp + fp + 1e-9); r = tp / (tp + fn + 1e-9)
    return dict(IoU=iou, F1=2*p*r/(p+r+1e-9), precision=p, recall=r, OA=(tp+tn)/(tp+tn+fp+fn+1e-9))

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); acc = {k: np.zeros(4) for k in ['all', 'flat <5°', 'sloped ≥5°']}
    for x, y, s in loader:
        p = model(x.to(DEV)).argmax(1).cpu(); valid = y != 255
        for k, m in [('all', valid), ('flat <5°', valid & (s < 5)), ('sloped ≥5°', valid & (s >= 5))]:
            pp, yy = p[m], y[m]
            acc[k] += [((pp == 1) & (yy == 1)).sum(), ((pp == 1) & (yy == 0)).sum(),
                       ((pp == 0) & (yy == 1)).sum(), ((pp == 0) & (yy == 0)).sum()]
    return {k: scores(*v) for k, v in acc.items()}

def run(name, ch, seed, attention, epochs=60):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    dl = {k: DataLoader(Chips(v, ch, train=(k == 'train')), batch_size=16 if k == 'train' else 4,
                        shuffle=(k == 'train'), num_workers=2) for k, v in split.items()}
    model = smp.Unet('resnet34', encoder_weights=None, in_channels=len(ch), classes=2,
                     decoder_attention_type=attention).to(DEV)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1., 3.], device=DEV), ignore_index=255)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, path = 0, f'{OUT}/runs/unet_{name}_seed{seed}.pt'
    for ep in range(epochs):
        model.train()
        for x, y, _ in dl['train']:
            opt.zero_grad(); loss_fn(model(x.to(DEV)), y.to(DEV)).backward(); opt.step()
        sched.step()
        v = evaluate(model, dl['valid'])['all']['IoU']
        if v > best: best = v; torch.save(model.state_dict(), path)
        if ep % 10 == 0 or ep == epochs - 1: print(f'[{name} seed {seed}] epoch {ep:2d}  val IoU {v:.3f}')
    model.load_state_dict(torch.load(path))
    rows = [{'model': name, 'seed': seed, 'split': k, 'terrain': t, **m}
            for k in ['test', 'bolivia'] for t, m in evaluate(model, dl[k]).items()]
    return pd.DataFrame(rows)

## 3. Run all seeds (skips runs already saved)

In [ ]:
CONFIGS = {'C_att_sar': ([0, 1], 'scse'), 'D_att_sar_terrain': ([0, 1, 2, 3], 'scse')}
for seed in SEEDS:
    for name, (ch, att) in CONFIGS.items():
        f = f'{OUT}/runs/{name}_seed{seed}.csv'
        if os.path.exists(f): print('done already:', name, seed); continue
        run(name, ch, seed, att).to_csv(f, index=False)
        print('saved', f)

## 4. Results: mean ± std over seeds

In [ ]:
res = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{OUT}/runs/*.csv') + glob.glob(f'{PREV}/*.csv'))])
print('runs per model:', res.groupby('model').seed.nunique().to_dict())
agg = (res.groupby(['split', 'terrain', 'model'])['IoU'].agg(['mean', 'std', 'count'])
          .round(3).reset_index())
table = agg.assign(IoU=lambda d: d['mean'].map('{:.3f}'.format) + ' ± ' + d['std'].map('{:.3f}'.format))
table = table.pivot_table(index=['split', 'terrain'], columns='model', values='IoU', aggfunc='first')
table.to_csv(f'{OUT}/iou_mean_std.csv'); table

## 5. Paired gains (same seed)
- **attention effect:** C − A (SAR only) and D − B (with terrain)
- **terrain effect with attention:** D − C

In [ ]:
w = res.pivot_table(index=['split', 'terrain', 'seed'], columns='model', values='IoU').reset_index()
pairs = {'attention, SAR (C − A)': ('C_att_sar', 'A_sar'),
         'attention, +terrain (D − B)': ('D_att_sar_terrain', 'B_sar_terrain'),
         'terrain, attention (D − C)': ('D_att_sar_terrain', 'C_att_sar')}
out = []
for label, (hi, lo) in pairs.items():
    g = w.assign(gain=w[hi] - w[lo]).groupby(['split', 'terrain'])['gain']
    out.append(g.agg(mean='mean', std='std', seeds_positive=lambda x: f'{(x > 0).sum()}/{len(x)}')
                .round(3).assign(comparison=label).reset_index())
gains = pd.concat(out)
gains = gains[gains.split == 'test'].set_index(['comparison', 'terrain'])
gains.to_csv(f'{OUT}/paired_gains_attention.csv'); gains